# AlphaFold3 Structure Prediction

![AlphaFold3 Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/alphafold3/hero.png)

This notebook demonstrates multi-modal structure prediction using AlphaFold3 from Google DeepMind. Unlike its predecessor, AlphaFold3 jointly predicts the structures of proteins, nucleic acids (DNA and RNA), ligands, and their complexes using a diffusion-based architecture that generates multiple structural samples per seed. We demonstrate `run_alphafold3` by predicting the structure of the MfnG protein bound to L-tyrosine, a practical example of protein-ligand complex prediction where both protein fold and small molecule geometry must be resolved simultaneously.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("alphafold3")
display_overview("alphafold3")
display_docs_section("alphafold3", "Background")

# AlphaFold3

AlphaFold3 (AF3) is Google DeepMind and Isomorphic Labs' third-generation AlphaFold, extending structure prediction beyond single proteins to the joint 3D structure of proteins together with DNA, RNA, and small-molecule ligands in one model. This toolkit runs AlphaFold3 so you can fold these mixed-molecule complexes from sequence through the proto framework, provided you have access to the gated model weights on your machine.

AlphaFold3 ([Abramson et al., 2024](https://doi.org/10.1038/s41586-024-07487-w)) predicts the joint 3D structure of a biomolecular assembly from the sequences and chemical components it contains. It extends AlphaFold2 beyond single proteins: one model folds complexes that mix proteins, DNA, RNA, and small-molecule ligands, and predicts how those parts are arranged relative to one another. As in AlphaFold2, each protein chain is paired with a multiple-sequence alignment (MSA) of related sequences, whose covariation patterns give the model an evolutionary signal for placing residues.

Internally, AlphaFold3 represents the assembly as a set of tokens: one per amino-acid residue or nucleotide, and one per atom for ligands and modified residues. It then learns a representation of every token and of every token pair. Where AlphaFold2 leaned on the large MSA-centric Evoformer, AlphaFold3 de-emphasizes the MSA, handling it in a separate preliminary module rather than iterating it through the deep trunk, and does most of its work in the 'Pairformer', which iteratively refines the token and pair representations through geometry-inspired "triangle attention" updates. The final representations are then fed into a diffusion module that iteratively denoises all-atom coordinates starting from random noise. Run from several random seeds, it produces multiple candidate structures, and the highest-confidence candidate is returned as the final prediction. In addition, AlphaFold3 reports calibrated confidence metrics such as the per-atom predicted local distance difference test (pLDDT) for local reliability, a predicted aligned error (PAE) for how well any two tokens are placed relative to each other, and predicted template-modeling (pTM) and interface predicted template-modeling (ipTM) scores for overall and interface accuracy.

## Available tools

In [2]:
display_available_tools("alphafold3")

- **`run_alphafold3()`** — Protein structure prediction using AlphaFold3

### `run_alphafold3`

AlphaFold3 extends structure prediction beyond proteins to jointly model proteins, DNA, RNA, ligands, and their complexes in a single unified framework. Its diffusion-based architecture generates five structural samples per seed and automatically selects the highest-confidence prediction. SMILES strings for small molecules are automatically converted to CCD (Chemical Component Dictionary) codes internally. The `seeds` parameter controls stochastic sampling and can be extended to multiple values for more thorough exploration of conformational space, which is particularly useful for challenging tasks such as antibody-antigen docking.

In [3]:
from pathlib import Path

from proto_tools import (
    AlphaFold3Config,
    AlphaFold3Input,
    Chain,
    Complex,
    run_alphafold3,
)

In [4]:
# Display input docs
display_api_reference("alphafold3", "input", "run_alphafold3")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = Complex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = AlphaFold3Input(complexes=[complex])

**Input** — `AlphaFold3Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `list[proto_tools.entities.complex.Complex]` | required | List of complexes to predict structure for containing chains and entity types. |
| `msas` | `list[proto_tools.tools.structure_prediction.shared_data_models.ComplexMSAs] \| None` | `None` | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [5]:
# Display config docs
display_api_reference("alphafold3", "config", "run_alphafold3")

# Configure AlphaFold3
config = AlphaFold3Config(
    name="mfng_ligand",
    seeds=[0],  # AlphaFold3 generates 5 diffusion samples per seed
    use_msa=False,
    verbose=False,
)

**Config** — `AlphaFold3Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on (e.g., 'cuda', 'cpu') |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `include_pae_matrix` | `bool` | `False` | Attach the full per-residue PAE matrix. |
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains using MMseqs2 homology search |
| `msa_search_config` | `proto_tools.tools.sequence_alignment.mmseqs2.homology_search.Mmseqs2HomologySearchConfig \| None` | `None` | Nested MMseqs2 homology-search config for MSA generation; None uses default settings. |
| `name` | `str` | `'af3_job'` | Name of the AlphaFold3 folding job |
| `seeds` | `list[int]` | `[0]` | Seeds to use for AlphaFold3 when the common seed field is unset. |
| `output_dir` | `str \| None` | `None` | Prefix for the AlphaFold3 output directory. If None, uses temp directory with auto-cleanup. |
| `model_dir` | `str \| None` | `None` | Directory with AlphaFold3 weights. If unset, resolves from env vars. |
| `sif_path` | `str \| None` | `None` | Pre-built AlphaFold3 .sif image. If unset, prefers provisioned sif then env. |
| `num_recycles` | `int` | `10` | Recycling iterations through the model. Higher = more accurate but slower. |
| `num_diffusion_samples` | `int` | `5` | Diffusion samples per seed; best by ranking score is kept. Total = len(seeds) x samples. |

In [6]:
# Run structure prediction
result = run_alphafold3(inputs, config)

Folding structures (AlphaFold3):   0%|          | 0/1 [00:00<?, ?complex/s]

In [7]:
# Display output docs
display_api_reference("alphafold3", "output", "run_alphafold3")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Ranking score: {mfng_structure.metrics.ranking_score:.3f}")
print(f"  Average pLDDT: {mfng_structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.metrics.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.metrics.iptm:.3f}")

**Output** — `AlphaFold3Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `list[proto_tools.entities.structures.structure.Structure]` | required | List of predicted structures, one per input complex. |

  Number of chains: 2
  Protein length: 384 residues
  Ranking score: 0.410
  Average pLDDT: 35.307
  pTM score: 0.260
  ipTM score: 0.430


#### Visualize the predicted complex

The interactive viewer renders the predicted protein-ligand complex colored by chain. The protein backbone is shown alongside the bound L-tyrosine ligand, allowing you to inspect the predicted binding pose and overall fold quality.

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [8]:
mfng_structure.visualize(style='stick', color_by='chain')

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT scores for per-residue confidence visualization.

In [9]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'mfng_ligand_complex.pdb'}")

Structure exported to example_output/mfng_ligand_complex.pdb
